# Week 1 – Day 1-2: Database Setup & CSV Ingestion
Loads all 8 Olist CSVs into a local PostgreSQL database with proper schema and foreign keys.

## 1. Configuration
Edit the connection string below to match your PostgreSQL credentials.

In [4]:
# ── dependencies ──────────────────────────────────────────────────────────────
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# ── config ────────────────────────────────────────────────────────────────────
DB_USER     = "olist_db"       # change if different
DB_PASSWORD = "1234"       # change to your password
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "ecommerce_db"

DATA_DIR = Path("Data")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    echo=False
)
print("Engine created. DB:", DB_NAME)

Engine created. DB: ecommerce_db


## 2. Create Database (run once from psql or pgAdmin)
```sql
CREATE DATABASE ecommerce_db;
```
Then come back here and run the cells below.

## 3. Define Schema (DDL)
Creates all 9 tables with correct column types, primary keys, and foreign keys.

In [5]:
DDL = """
-- drop in FK-safe order
DROP TABLE IF EXISTS order_reviews        CASCADE;
DROP TABLE IF EXISTS order_payments       CASCADE;
DROP TABLE IF EXISTS order_items          CASCADE;
DROP TABLE IF EXISTS orders               CASCADE;
DROP TABLE IF EXISTS customers            CASCADE;
DROP TABLE IF EXISTS sellers              CASCADE;
DROP TABLE IF EXISTS products             CASCADE;
DROP TABLE IF EXISTS product_category_translation CASCADE;
DROP TABLE IF EXISTS geolocation          CASCADE;

-- ── dimension tables ───────────────────────────────────────────────────────
CREATE TABLE geolocation (
    geolocation_zip_code_prefix  VARCHAR(10),
    geolocation_lat              FLOAT,
    geolocation_lng              FLOAT,
    geolocation_city             VARCHAR(100),
    geolocation_state            CHAR(2)
);

CREATE TABLE product_category_translation (
    product_category_name         VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
);

CREATE TABLE products (
    product_id                   VARCHAR(50)  PRIMARY KEY,
    product_category_name        VARCHAR(100),
    product_name_length          INT,
    product_description_length   INT,
    product_photos_qty           INT,
    product_weight_g             FLOAT,
    product_length_cm            FLOAT,
    product_height_cm            FLOAT,
    product_width_cm             FLOAT
);

CREATE TABLE sellers (
    seller_id             VARCHAR(50)  PRIMARY KEY,
    seller_zip_code_prefix VARCHAR(10),
    seller_city           VARCHAR(100),
    seller_state          CHAR(2)
);

CREATE TABLE customers (
    customer_id               VARCHAR(50) PRIMARY KEY,
    customer_unique_id        VARCHAR(50),
    customer_zip_code_prefix  VARCHAR(10),
    customer_city             VARCHAR(100),
    customer_state            CHAR(2)
);

-- ── fact tables ─────────────────────────────────────────────────────────────
CREATE TABLE orders (
    order_id                        VARCHAR(50) PRIMARY KEY,
    customer_id                     VARCHAR(50) REFERENCES customers(customer_id),
    order_status                    VARCHAR(20),
    order_purchase_timestamp        TIMESTAMP,
    order_approved_at               TIMESTAMP,
    order_delivered_carrier_date    TIMESTAMP,
    order_delivered_customer_date   TIMESTAMP,
    order_estimated_delivery_date   TIMESTAMP
);

CREATE TABLE order_items (
    order_id             VARCHAR(50) REFERENCES orders(order_id),
    order_item_id        INT,
    product_id           VARCHAR(50) REFERENCES products(product_id),
    seller_id            VARCHAR(50) REFERENCES sellers(seller_id),
    shipping_limit_date  TIMESTAMP,
    price                NUMERIC(10,2),
    freight_value        NUMERIC(10,2),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE TABLE order_payments (
    order_id              VARCHAR(50) REFERENCES orders(order_id),
    payment_sequential    INT,
    payment_type          VARCHAR(30),
    payment_installments  INT,
    payment_value         NUMERIC(10,2),
    PRIMARY KEY (order_id, payment_sequential)
);

CREATE TABLE order_reviews (
    review_id               VARCHAR(50),
    order_id                VARCHAR(50) REFERENCES orders(order_id),
    review_score            SMALLINT,
    review_comment_title    TEXT,
    review_comment_message  TEXT,
    review_creation_date    TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    PRIMARY KEY (review_id, order_id)
);
"""

with engine.connect() as conn:
    conn.execute(text(DDL))
    conn.commit()

print("Schema created successfully.")

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: FATAL:  password authentication failed for user "olist_db"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 4. Load CSVs

In [ ]:
# helper: parse timestamp columns that may be empty
def load_csv(filename, parse_dates=None):
    path = DATA_DIR / filename
    df = pd.read_csv(path, parse_dates=parse_dates)
    print(f"  {filename}: {len(df):,} rows, {df.shape[1]} cols")
    return df

print("Loading CSVs...")
df_geo          = load_csv("olist_geolocation_dataset.csv")
df_cat          = load_csv("product_category_name_translation.csv")
df_products     = load_csv("olist_products_dataset.csv")
df_sellers      = load_csv("olist_sellers_dataset.csv")
df_customers    = load_csv("olist_customers_dataset.csv")
df_orders       = load_csv("olist_orders_dataset.csv",
                           parse_dates=["order_purchase_timestamp",
                                        "order_approved_at",
                                        "order_delivered_carrier_date",
                                        "order_delivered_customer_date",
                                        "order_estimated_delivery_date"])
df_items        = load_csv("olist_order_items_dataset.csv",
                           parse_dates=["shipping_limit_date"])
df_payments     = load_csv("olist_order_payments_dataset.csv")
df_reviews      = load_csv("olist_order_reviews_dataset.csv",
                           parse_dates=["review_creation_date",
                                        "review_answer_timestamp"])
print("Done.")

In [ ]:
# quick shape check
datasets = {
    "geolocation": df_geo,
    "products": df_products,
    "sellers": df_sellers,
    "customers": df_customers,
    "orders": df_orders,
    "order_items": df_items,
    "order_payments": df_payments,
    "order_reviews": df_reviews,
}
pd.DataFrame([(k, v.shape[0], v.shape[1]) for k, v in datasets.items()],
             columns=["table", "rows", "cols"])

## 5. Write to PostgreSQL
Order matters — dimension tables first, then fact tables (FK constraints).

In [ ]:
def insert(df, table, chunksize=10_000):
    df.to_sql(table, engine, if_exists="append", index=False, chunksize=chunksize, method="multi")
    print(f"  ✓ {table}: {len(df):,} rows inserted")

print("Inserting into PostgreSQL...")

# dimensions (no FK deps)
insert(df_geo,       "geolocation")
insert(df_cat,       "product_category_translation")
insert(df_products,  "products")
insert(df_sellers,   "sellers")
insert(df_customers, "customers")

# facts (depend on dimensions)
insert(df_orders,   "orders")
insert(df_items,    "order_items")
insert(df_payments, "order_payments")
insert(df_reviews,  "order_reviews")

print("\nAll tables loaded.")

## 6. Validation

In [ ]:
validation_sql = """
SELECT 'geolocation'               AS tbl, COUNT(*) AS rows FROM geolocation
UNION ALL SELECT 'products',                COUNT(*) FROM products
UNION ALL SELECT 'sellers',                 COUNT(*) FROM sellers
UNION ALL SELECT 'customers',               COUNT(*) FROM customers
UNION ALL SELECT 'orders',                  COUNT(*) FROM orders
UNION ALL SELECT 'order_items',             COUNT(*) FROM order_items
UNION ALL SELECT 'order_payments',          COUNT(*) FROM order_payments
UNION ALL SELECT 'order_reviews',           COUNT(*) FROM order_reviews
ORDER BY tbl;
"""

with engine.connect() as conn:
    result = pd.read_sql(text(validation_sql), conn)

result

In [ ]:
# null check on critical FK columns
null_checks = {
    "orders.customer_id nulls": "SELECT COUNT(*) FROM orders WHERE customer_id IS NULL",
    "order_items.order_id nulls": "SELECT COUNT(*) FROM order_items WHERE order_id IS NULL",
    "order_items.product_id nulls": "SELECT COUNT(*) FROM order_items WHERE product_id IS NULL",
}

with engine.connect() as conn:
    for label, q in null_checks.items():
        n = conn.execute(text(q)).scalar()
        status = "OK" if n == 0 else f"WARNING: {n} nulls"
        print(f"  {label}: {status}")

In [ ]:
# sanity: peek at joined data
sample_sql = """
SELECT o.order_id,
       c.customer_state,
       o.order_status,
       o.order_purchase_timestamp,
       p.payment_value,
       r.review_score
FROM orders o
JOIN customers      c ON c.customer_id  = o.customer_id
JOIN order_payments p ON p.order_id     = o.order_id
LEFT JOIN order_reviews r ON r.order_id = o.order_id
LIMIT 10;
"""

with engine.connect() as conn:
    pd.read_sql(text(sample_sql), conn)

In [ ]:
\COPY customers FROM 'D:\Programming\Projects\E-commerce_project\Data\olist_customers_dataset.csv' DELIMITER ',' CSV HEADER;
\COPY orders FROM 'D:\Programming\Projects\E-commerce_project\Data\olist_orders_dataset.csv' DELIMITER ',' CSV HEADER;
\COPY order_items FROM 'D:\Programming\Projects\E-commerce_project\Data\olist_order_items_dataset.csv' DELIMITER ',' CSV HEADER;
\COPY payments FROM 'D:\Programming\Projects\E-commerce_project\Data\olist_order_payments_dataset.csv' DELIMITER ',' CSV HEADER;
\COPY reviews FROM 'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv' DELIMITER ',' CSV HEADER;

In [7]:
import pandas as pd

df = pd.read_csv(
    r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv',
    encoding='latin1',
    encoding_errors='replace'
)

df.to_csv(r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv', index=False, encoding='utf8')

In [8]:
with open(r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv', 'rb') as f:
    content = f.read()

content = content.decode('latin1').encode('utf8')

with open(r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_clean.csv', 'wb') as f:
    f.write(content)

In [1]:
with open(r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv', 'rb') as f:
    content = f.read()

cleaned = content.decode('utf-8', errors='replace')

with open(r'D:\Programming\Projects\E-commerce_project\Data\olist_order_reviews_dataset.csv', 'w', encoding='utf-8') as f:
    f.write(cleaned)